<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-30</br>
</div>

</br>

In [ ]:
# TODO 0: 디바이스 설정, 모델/토크나이저 로드, 데이터셋을 준비해봅시다.

import torch, warnings
warnings.filterwarnings("ignore")

# 디바이스 확인
if torch.cuda.is_available():
    print(f"✅ CUDA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("⚠️ MPS 감지")
else:
    print("⚠️ GPU 미감지")

# 모델 + 토크나이저 로드
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
print(f"✅ 모델 로드 완료: {model_name}")

# 데이터셋 로드
from datasets import load_dataset
import datasets

dataset = load_dataset("shangrilar/ko_text2sql", data_dir="origin", split="train")
print(f"✅ 데이터셋 로드 완료: {len(dataset)}개 샘플")
print(f"   컬럼: {list(dataset.column_names)}")
print(f"   예시: {dataset[0]}")

</br>

# 학습 내용
>이번 장에서는 <strong>LoRA 적용 및 학습(LoRA Application & Training)</strong>에 대해 학습합니다.</br></br>
>ChatML 형식으로 데이터를 변환하고 SFTTrainer로 LoRA 학습을 학습해봅시다.

</br>

# ChatML 데이터 변환 (ChatML Conversion)
> 학습 데이터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ChatML 형식</mark>으로 변환하여 대화형 파인튜닝에 적합하게 만듭니다.

> 실제 LoRA를 적용할 때는 두 가지 핵심 결정이 필요합니다.</br>
> 첫째, **Rank(r) 선택**입니다. rank가 높을수록 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">표현력이 올라가지만 메모리 사용량도 증가</mark>하며, 일반적으로 r=8~16이 균형점입니다.</br>
> 둘째, **Target Modules 선택**으로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 레이어에 LoRA를 적용할지</mark> 결정합니다.</br>
> Attention 레이어(`q_proj`, `v_proj`)만 적용하면 파라미터는 적고, MLP 레이어까지 포함하면 성능은 올라가지만 파라미터 수도 늘어납니다.</br>
> 잘못된 설정은 학습 자원 낭비로 이어지므로, 목적에 맞는 설정 탐색이 중요합니다.

In [ ]:
# TODO 1: 시스템 프롬프트와 사용자 프롬프트 템플릿을 정의해봅시다.


In [ ]:
# TODO 2: ChatML 변환 함수를 정의해봅시다.


In [ ]:
# TODO 3: 데이터셋을 변환해봅시다.


</br>

## ChatML 형식 예시

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">토큰</th>
      <th>역할</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>&lt;|im_start|&gt;</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">메시지 시작</mark></td></tr>
    <tr><td style="text-align:center"><code>&lt;|im_end|&gt;</code></td><td>메시지 종료</td></tr>
    <tr><td style="text-align:center"><code>system/user/assistant</code></td><td>역할 구분</td></tr>
  </tbody>
</table>

</br>

# SFTTrainer (Supervised Fine-Tuning)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Hugging Face TRL 라이브러리</mark>의 SFT 전용 트레이너입니다.

In [ ]:
# TODO 4: LoraConfig를 사용하여 r=16, lora_alpha=32, target_modules를 설정해봅시다.


In [ ]:
# TODO 5: SFTConfig를 설정해봅시다.


In [ ]:
# TODO 6: SFTTrainer를 생성해봅시다. (peft_config을 직접 전달 → 내부에서 get_peft_model 처리)


</br>

## 주요 학습 설정

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">권장값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>learning_rate</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습률</mark></td><td style="text-align:center">2e-4 ~ 5e-5</td></tr>
    <tr><td style="text-align:center"><code>num_train_epochs</code></td><td>에폭 수</td><td style="text-align:center">1~3</td></tr>
    <tr><td style="text-align:center"><code>per_device_train_batch_size</code></td><td>배치 크기</td><td style="text-align:center">2~8</td></tr>
    <tr><td style="text-align:center"><code>max_seq_length</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">최대 시퀀스 길이</mark></td><td style="text-align:center">512~2048</td></tr>
    <tr><td style="text-align:center"><code>dataset_text_field</code></td><td>텍스트 필드명</td><td style="text-align:center">"text"</td></tr>
  </tbody>
</table>

💡Unsloth 가속
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Unsloth</mark>는 LoRA 학습을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">2~5배 가속</mark>하는 라이브러리입니다.</br>
> `FastLanguageModel.from_pretrained()`로 모델을 로드하면 자동으로 최적화됩니다.

💡max_seq_length 설정
> 데이터의 최대 길이보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">짧으면 잘림</mark>이 발생합니다.</br>
> VRAM이 허용하는 범위에서 충분히 크게 설정하세요.

</br>

# Label Masking과 train_on_responses_only
> 학습 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모델의 응답(response) 부분만 loss를 계산</mark>하고, 지시문(instruction) 부분은 무시하는 기법입니다.

> SFT(Supervised Fine-Tuning)에서 전체 시퀀스에 대해 loss를 계산하면, 모델은 **사용자의 질문까지 "외우려고"** 합니다.</br></br>
> 이는 비효율적이며 과적합의 원인이 됩니다.</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Label Masking</mark>은 instruction 부분의 label을 **-100**으로 설정하여 `CrossEntropyLoss`가 해당 토큰을 무시하게 만드는 기법입니다. 이를 통해 모델은 **"답변을 잘 생성하는 것"**에만 집중하여 학습합니다.

## Label Masking 동작 원리

<div style="text-align:center">

</div>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구분</th>
      <th style="text-align:center">Label 값</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Instruction 부분</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">-100</mark></td><td><code>CrossEntropyLoss(ignore_index=-100)</code>에 의해 loss 계산에서 제외</td></tr>
    <tr><td style="text-align:center">Response 부분</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제 토큰 ID</mark></td><td>모델이 예측해야 하는 정답 토큰 — loss 계산 대상</td></tr>
  </tbody>
</table>

## Unsloth의 train_on_responses_only

> Unsloth(및 TRL)에서는 `train_on_responses_only` 함수로 Label Masking을 간편하게 적용할 수 있습니다.

```python
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",       # instruction 시작 구분자
    response_part="<|im_start|>assistant\n",      # response 시작 구분자
)
```

<mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">`instruction_part`</mark>부터 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">`response_part`</mark> 직전까지의 토큰이 **-100으로 마스킹**됩니다.</br></br>
구분자 문자열은 사용하는 chat template에 따라 달라지며, Qwen2.5의 경우 위와 같이 설정합니다.

In [ ]:
# TODO 7: train_on_responses_only를 적용하여 response 부분만 학습하도록 설정해봅시다.


In [ ]:
# TODO 8: 학습을 실행하고 모델을 저장해봅시다.


In [ ]:
# TODO 9: trainer의 첫 번째 샘플에서 input_ids와 labels를 꺼내봅시다.


In [ ]:
# TODO 10: labels에서 -100(마스킹)과 실제 토큰 ID의 비율을 확인해봅시다.


In [ ]:
# TODO 11: 마스킹 경계를 시각화하여 MASKED → LEARN 전환 지점을 확인해봅시다.


💡왜 -100인가?
> PyTorch의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">`nn.CrossEntropyLoss`</mark>는 기본적으로 `ignore_index=-100`이 설정되어 있습니다.</br>
> label이 -100인 위치는 loss 계산에서 자동으로 제외되므로, 별도의 mask 텐서 없이도 특정 토큰을 학습에서 빼낼 수 있습니다.

💡Label Masking 없이 학습하면?
> instruction 부분까지 loss에 포함되면 모델이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">"질문을 외우는 데"</mark> 파라미터 용량을 낭비합니다.</br>
> 특히 instruction이 긴 경우(few-shot 프롬프트 등) 비효율이 더욱 심해지며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">response-only 학습이 일반적으로 더 높은 성능</mark>을 보입니다.

</br>

# 학습 전후 비교 (Before vs After)
> 파인튜닝의 효과를 확인하기 위해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 입력에 대한 학습 전/후 응답</mark>을 비교합니다.

학습 전 베이스 모델의 응답을 미리 저장해 두고, 학습 후 동일 입력으로 추론하여 비교하면 파인튜닝 효과를 직관적으로 확인할 수 있습니다. 비교 시에는 반드시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습에 사용하지 않은 데이터</mark>로 테스트해야 공정한 평가가 가능합니다.

In [ ]:
# TODO 12: 테스트용 프롬프트를 정의하고 ChatML 형식으로 변환해봅시다.


In [ ]:
# TODO 13: 학습 전 베이스 모델의 응답을 생성하고 before_response에 저장해봅시다.


In [ ]:
# TODO 14: 학습 후 동일한 프롬프트로 추론하여 after_response를 생성해봅시다.


In [ ]:
# TODO 15: 학습 전후 응답을 비교해봅시다.


💡학습 전후 비교 팁
> 비교 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">temperature=0.0</mark>으로 설정하면 deterministic한 응답을 얻을 수 있어 더 정확한 비교가 가능합니다.</br>
> 학습 전 응답을 변수에 저장해 두지 않으면 학습 후 비교가 불가능하므로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 전에 반드시 TODO 5를 먼저 실행</mark>하세요.

</br>

</br>

## LoRA 핵심 하이퍼파라미터

💡lora_alpha 역할
> `lora_alpha`는 LoRA 출력에 곱해지는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">스케일 계수</mark>입니다.</br>
> 실제 적용 스케일 = `alpha / r` — `r`이 커질 때 alpha도 같이 올려 균형을 맞춥니다.

💡target_modules 선택 기준
> Attention 레이어(`q_proj`, `k_proj`, `v_proj`, `o_proj`)에 적용하는 것이 일반적으로 효과적입니다.</br>
> MLP 레이어(`gate_proj`, `up_proj`, `down_proj`)까지 포함하면 성능이 오르지만 파라미터 수도 증가합니다.